# Simple PySaprk MLlib Experiment

## Imports Essential Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.stat import Correlation
import pyspark.sql.functions as f
from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import *

- Imports `SparkSession`, the entry point for working with Spark SQL and DataFrames.
- Imports `correlation` utilities from MLlib (not used in this script, but available if needed).
- Imports Spark SQL functions and aliases them as f.
- Imports the `Vector` type used in MLlib for numeric machine learning features.
- Imports `VectorAssembler`, the tool that converts columns into a single features vector required by MLlib.

## Schema Definition and Data Loading

In [2]:
Schema = StructType([
    StructField('YearsExperience', FloatType(), True),
    StructField('Salary', FloatType(), True)
])

Defines a custom schema for the CSV: `YearsExperience` and `Salary`

- Both as float numbers
- Both nullable (True)

In [3]:

spark = SparkSession.builder \
    .appName("Read HDFS Example") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

df = spark.read.csv("/linkedin-data/Salary_Data.csv", header=True, schema=Schema)

Reads the CSV file into a DataFrame:
- `header=True` → uses the first row as column names
- `schema=Schema` → forces Spark to use the provided schema

In [4]:
df.head(5)

[Row(YearsExperience=1.100000023841858, Salary=39343.0),
 Row(YearsExperience=1.2999999523162842, Salary=46205.0),
 Row(YearsExperience=1.5, Salary=37731.0),
 Row(YearsExperience=2.0, Salary=43525.0),
 Row(YearsExperience=2.200000047683716, Salary=39891.0)]

Show dataframe with associated schema

## Feature Engineering

In [5]:
assembler = VectorAssembler(inputCols=['YearsExperience'],outputCol='features')

Creates a VectorAssembler that transforms the `YearsExperience` column into a MLlib-compatible `features` vector (e.g., `[3.0]`).

In [6]:
data_set = assembler.transform(df)

Applies the `VectorAssembler` to the DataFrame, producing a new column called `features`.

In [7]:
data_set = data_set.select(['features','salary'])

Selects only the columns needed by the MLlib model:

- `features` (input)
- `salary` (label / target)

## Train/Test Split

In [8]:
train_data,test_data = data_set.randomSplit([0.8,0.2])

Splits the dataset into:

- 80% training data
- 20% testing data

This ensures the model is trained and evaluated on separate data.

## Model Training

In [9]:
from pyspark.ml.regression import LinearRegression

In [10]:
lr = LinearRegression(featuresCol="features",labelCol='salary')

In [11]:
lrModel = lr.fit(train_data)

## Model Evaluation

In [12]:
test_stats = lrModel.evaluate(test_data)

Evaluates the trained model on the testing dataset and returns metrics.

## Print Evaluation Metrics

In [13]:
print(f"RMSE: {test_stats.rootMeanSquaredError}")
print(f"R2: {test_stats.r2}")
print(f"MSE: {test_stats.meanSquaredError}")

RMSE: 4791.526392938719
R2: 0.9796179443044208
MSE: 22958725.174228337


- Prints RMSE (Root Mean Squared Error) — lower is better.
- Prints R² score — closer to 1 is better.
- Prints MSE (Mean Squared Error) — lower is better